# 09 — Walk-Forward Validation

**Purpose:** Out-of-sample and walk-forward testing — the primary evidence standard for this project (mandate §4.5).

**Research questions:**
1. Does performance persist in untouched OOS data?
2. Do rolling and anchored walk-forwards both pass the consistency acceptance rules?
3. Are reoptimized parameters stable across folds, or do they thrash?

**Data used:** full history, split per walk_forward_config.yaml (**blocked**, L-001).


In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
print(f"config loaded | global seed = {SEED}")


## Methodology

`walk_forward.make_windows` (both schemes, 5-day embargo) + `run_walk_forward`: fit reoptimizes only the whitelisted parameters on train, evaluates frozen on test; `acceptance_check` enforces ≥55% profitable folds, ≤40% single-window PnL share, positive total after stressed costs; parameter-path stability plots; per-fold results appended to the experiment log.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_AVAILABLE = any(DATA_DIR.glob("*_minute.*")) if DATA_DIR.exists() else False
if not DATA_AVAILABLE:
    print("BLOCKED-ON-DATA: no futures market data in this environment (see "
          "reports/00_repository_audit.md, issue L-001).\n"
          "Run this notebook inside QuantConnect Research, or drop licensed data\n"
          "into data/processed/ in the canonical schema (src/spread_research/data_loader.py).")


In [ ]:
if DATA_AVAILABLE:
    from spread_research.walk_forward import make_windows, run_walk_forward, acceptance_check
    print("wire walk-forward here")

## Results

**BLOCKED-ON-DATA** — this section intentionally contains no results. No synthetic or fabricated market findings are presented as evidence (CLAUDE.md gate 3). It will be populated when the notebook runs against real data.

## Limitations

~7 years of micro history yields ~20 quarterly folds — modest power; acceptance thresholds are set for consistency, not significance theater.

## Decision

Pairs/configs failing acceptance are rejected regardless of full-sample Sharpe. Parameter thrash across folds is treated as overfitting evidence.

## What this means for the algorithm

Only walk-forward survivors are candidates for the final go recommendation; their fold-median (not best-fold) performance is the number reported forward.